In [1]:
import os
import json
import joblib
import pandas as pd
import numpy as np
try:
    import xgboost as xgb
except:
    !pip install "xgboost<3"
    import xgboost as xgb
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import log_loss, accuracy_score

/home/ec2-user/anaconda3/envs/python3/lib/python3.12/site-packages/xgboost/core.py:265: FutureWarning: Your system has an old version of glibc (< 2.28). We will stop supporting Linux distros with glibc older than 2.28 after **May 31, 2025**. Please upgrade to a recent Linux distro (with glibc 2.28+) to use future versions of XGBoost.
Note: You have installed the 'manylinux2014' variant of XGBoost. Certain features such as GPU algorithms or federated learning are not available. To use these features, please upgrade to a recent Linux distro with glibc 2.28+, and install the 'manylinux_2_28' variant.
  warnings.warn(


#### Constants

In [2]:
str_bucket = os.getcwd().split('/')[4].replace('_','-')
print(f'Bucket: {str_bucket}')

str_task = os.getcwd().split('/')[5]
print(f'Task: {str_task}')

str_dirname_output = './output'
os.makedirs(str_dirname_output, exist_ok=True)

Bucket: assessment-swish-analytics
Task: 04_model


#### Import Data from S3

In [3]:
X_train = pd.read_csv(f's3://{str_bucket}/03_preprocessing/X_train.csv')
X_valid = pd.read_csv(f's3://{str_bucket}/03_preprocessing/X_valid.csv')
X_test = pd.read_csv(f's3://{str_bucket}/03_preprocessing/X_test.csv')

y_train = pd.read_csv(f's3://{str_bucket}/03_preprocessing/y_train.csv').squeeze()
y_valid = pd.read_csv(f's3://{str_bucket}/03_preprocessing/y_valid.csv').squeeze()
y_test = pd.read_csv(f's3://{str_bucket}/03_preprocessing/y_test.csv').squeeze()

print(f'X_train: {X_train.shape}, y_train: {y_train.shape}')
print(f'X_valid: {X_valid.shape}, y_valid: {y_valid.shape}')
print(f'X_test:  {X_test.shape}, y_test:  {y_test.shape}')

/home/ec2-user/anaconda3/envs/python3/lib/python3.12/site-packages/fsspec/registry.py:301: UserWarning: Your installed version of s3fs is very old and known to cause
severe performance issues, see also https://github.com/dask/dask/issues/10276

To fix, you should specify a lower version bound on s3fs, or
update the current installation.

  warnings.warn(s3_msg)


X_train: (538294, 78), y_train: (538294,)
X_valid: (120479, 78), y_valid: (120479,)
X_test:  (39545, 78), y_test:  (39545,)


#### Encode Target Labels

In [4]:
le = LabelEncoder()
le.fit(y_train)

y_train_enc = le.transform(y_train)
y_valid_enc = le.transform(y_valid)
y_test_enc = le.transform(y_test)

print(f'Classes: {le.classes_}')
print(f'Number of classes: {len(le.classes_)}')

Classes: ['CH' 'CU' 'FC' 'FF' 'FS' 'FT' 'SI' 'SL']
Number of classes: 8


#### Create DMatrices

In [5]:
dtrain = xgb.DMatrix(X_train, label=y_train_enc)
dvalid = xgb.DMatrix(X_valid, label=y_valid_enc)
dtest = xgb.DMatrix(X_test, label=y_test_enc)

#### Train XGBoost Model

Multi-class classification with `multi:softprob` to get calibrated probability outputs for each pitch type.

In [ ]:
dict_params = {
    'objective': 'multi:softprob',
    'num_class': len(le.classes_),
    'eval_metric': 'mlogloss',
    'max_depth': 4,
    'learning_rate': 0.05,
    'subsample': 0.7,
    'colsample_bytree': 0.5,
    'min_child_weight': 5,
    'reg_lambda': 1.0,
    'gamma': 0.1,
    'seed': 42,
    'verbosity': 1,
}

print('Parameters:')
for k, v in dict_params.items():
    print(f'  {k}: {v}')

In [ ]:
# train with early stopping on validation set
model = xgb.train(
    dict_params,
    dtrain,
    num_boost_round=3000,
    evals=[(dtrain, 'train'), (dvalid, 'valid')],
    early_stopping_rounds=100,
    verbose_eval=100,
)

print(f'\nBest iteration: {model.best_iteration}')
print(f'Best validation mlogloss: {model.best_score:.4f}')

#### Quick Validation Check

In [8]:
# predictions on validation set
arr_valid_proba = model.predict(dvalid)
arr_valid_pred = le.classes_[arr_valid_proba.argmax(axis=1)]

flt_valid_acc = accuracy_score(y_valid, arr_valid_pred)
flt_valid_logloss = log_loss(y_valid_enc, arr_valid_proba)

print(f'Validation accuracy: {flt_valid_acc:.4f}')
print(f'Validation log-loss: {flt_valid_logloss:.4f}')

Validation accuracy: 0.2561
Validation log-loss: 2.0189


#### Save Model and Artifacts Locally

In [9]:
# save model locally
str_filename = 'xgb_model.json'
str_local_path = f'{str_dirname_output}/{str_filename}'
model.save_model(str_local_path)
print(f'Saved model to {str_local_path}')

# save label encoder locally
str_filename = 'label_encoder.joblib'
str_local_path = f'{str_dirname_output}/{str_filename}'
joblib.dump(le, str_local_path)
print(f'Saved label encoder to {str_local_path}')

# save test predictions locally
arr_test_proba = model.predict(dtest)
arr_test_pred = le.classes_[arr_test_proba.argmax(axis=1)]

df_test_preds = pd.DataFrame(arr_test_proba, columns=[f'prob_{c}' for c in le.classes_])
df_test_preds['predicted'] = arr_test_pred
df_test_preds['actual'] = y_test.values
str_filename = 'test_predictions.csv'
str_local_path = f'{str_dirname_output}/{str_filename}'
df_test_preds.to_csv(str_local_path, index=False)
print(f'Saved test predictions to {str_local_path}')

# save training params locally
dict_params['best_iteration'] = model.best_iteration
dict_params['best_score'] = model.best_score
str_filename = 'params.json'
str_local_path = f'{str_dirname_output}/{str_filename}'
with open(str_local_path, 'w') as f:
    json.dump(dict_params, f, indent=2)
print(f'Saved params to {str_local_path}')

Saved model to ./output/xgb_model.json
Saved label encoder to ./output/label_encoder.joblib
Saved test predictions to ./output/test_predictions.csv
Saved params to ./output/params.json
